In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from superlinked import framework as sl

from src.settings import Settings

settings = Settings()

/home/dinhln1/Desktop/hcmut_master/is_product_search/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# # Monkeypatch to handle sentence-transformers config differences across versions
# try:
#     from superlinked.framework.common.space.embedding.model_based.engine import sentence_transformers_engine as _ste
#     def _is_query_prompt_supported(self):
#         PROMPTS_KEY = "prompts"
#         QUERY_PROMPT_NAME = "query"
#         model = getattr(self, "_model", None)
#         if model is None:
#             return False
#         if hasattr(model, "_model_config"):
#             mc = getattr(model, "_model_config", {})
#             prompts_source = mc.get(PROMPTS_KEY, {}) if isinstance(mc, dict) else {}
#         elif hasattr(model, "_get_model_config"):
#             try:
#                 mc = model._get_model_config()
#                 prompts_source = mc.get(PROMPTS_KEY, {}) if isinstance(mc, dict) else {}
#             except Exception:
#                 prompts_source = {}
#         else:
#             prompts_source = getattr(model, "prompts", {})
#         return QUERY_PROMPT_NAME in (prompts_source or {})
#     _ste.SentenceTransformersEngine.is_query_prompt_supported = _is_query_prompt_supported
# except Exception as e:
#     print('Monkeypatch failed:', e)

In [4]:
class Product(sl.Schema):
    id: sl.IdField
    type: sl.String
    category: sl.String
    title: sl.String
    description: sl.String
    price: sl.Float
    review_rating: sl.Float


product = Product()

In [5]:
description_space = sl.TextSimilaritySpace(
    text=product.description,
    model="sentence-transformers/all-MiniLM-L6-v2",
)

title_space = sl.TextSimilaritySpace(
    text=product.title,
    model="sentence-transformers/all-MiniLM-L6-v2",
)

category_space = sl.CategoricalSimilaritySpace(
    category_input=product.category,
    categories=[
        "psychology",
        "children",
        "headphones",
        "electronics",
        "book",
    ],
)
# price
price_minimizer_space = sl.NumberSpace(
    number=product.price,
    min_value=0,
    max_value=200,
    mode=sl.Mode.MINIMUM,
)

# rating
rating_maximizer_space = sl.NumberSpace(
    number=product.review_rating,
    min_value=0,
    max_value=5,
    mode=sl.Mode.MAXIMUM,
)

In [6]:

# =========================================================
# 3. Build multi-attribute index
# =========================================================

product_index = sl.Index(
    spaces=[
        title_space,
        description_space,
        price_minimizer_space,
        rating_maximizer_space,
    ],
    fields=[
        product.type,
        product.category,
        product.title,
        product.price,
        product.review_rating,
    ],
)

In [7]:
source = sl.InMemorySource(product)

executor = sl.InMemoryExecutor(
    sources=[source],
    indices=[product_index],
)

app = executor.run()

In [8]:
products = [
    {
        "id": "1",
        "type": "book",
        "category": "psychology",
        "title": "Psychology Basics",
        "description": "introductory psychology book for beginners and students",
        "price": 30.0,
        "review_rating": 4.1,
    },
    {
        "id": "2",
        "type": "book",
        "category": "psychology",
        "title": "Advanced Psychology and Mindfulness",
        "description": "deep psychology, mindfulness, behavior and cognitive science",
        "price": 110.0,
        "review_rating": 4.9,
    },
    {
        "id": "3",
        "type": "book",
        "category": "children",
        "title": "Cheap Kids Learning Book",
        "description": "children learning book with basic emotional skills",
        "price": 12.0,
        "review_rating": 3.2,
    },
    {
        "id": "4",
        "type": "book",
        "category": "psychology",
        "title": "Mindfulness for Everyday Life",
        "description": "mindfulness, stress reduction, mental health and psychology",
        "price": 55.0,
        "review_rating": 4.7,
    },
    {
        "id": "5",
        "type": "electronics",
        "category": "headphones",
        "title": "Noise Cancelling Headphones",
        "description": "wireless headphones with active noise cancelling",
        "price": 80.0,
        "review_rating": 4.6,
    },
    {
        "id": "6",
        "type": "electronics",
        "category": "headphones",
        "title": "Wireless Earbuds Pro",
        "description": "compact earbuds with rich sound and long battery life",
        "price": 65.0,
        "review_rating": 4.5,
    },
    {
        "id": "7",
        "type": "electronics",
        "category": "electronics",
        "title": "Smart Watch Series 9",
        "description": "fitness tracking smartwatch with notifications and GPS",
        "price": 150.0,
        "review_rating": 4.8,
    },
    {
        "id": "8",
        "type": "electronics",
        "category": "electronics",
        "title": "Mechanical Keyboard",
        "description": "durable keyboard with tactile switches for typing and gaming",
        "price": 95.0,
        "review_rating": 4.4,
    },
    {
        "id": "9",
        "type": "electronics",
        "category": "electronics",
        "title": "4K Portable Monitor",
        "description": "thin portable monitor for laptops and mobile workstations",
        "price": 180.0,
        "review_rating": 4.3,
    },
    {
        "id": "10",
        "type": "electronics",
        "category": "electronics",
        "title": "Bluetooth Speaker",
        "description": "portable speaker with deep bass and waterproof design",
        "price": 45.0,
        "review_rating": 4.2,
    },
]

source.put(products)

In [9]:
openai_config = sl.OpenAIClientConfig(
    api_key=settings.OPENAI_API_KEY, model=settings.OPENAI_MODEL_ID
)

In [10]:
title_similar_param = sl.Param(
    "query_title",
    description=(
        "The text in the user's query that is used to search in the products' title."
        "Extract info that does not apply to other spaces or params."
    ),
)

description_similar_param = sl.Param(
    "query_description",
    description=(
        "The text in the user's query that is used to search in the products' description."
        " Extract info that does not apply to other spaces or params."
    ),
)

In [11]:
base_query = (
    sl.Query(
        product_index,
        weights={
            title_space: sl.Param("title_weight"),
            description_space: sl.Param("description_weight"),
            rating_maximizer_space: sl.Param(
                "review_rating_maximizer_weight"
            ),
            price_minimizer_space: sl.Param("price_minimizer_weights"),
        },
    )
    .find(product)
    .limit(sl.Param("limit"))
    .with_natural_query(sl.Param("natural_query"), openai_config)
    .filter(
        product.type
        == sl.Param(
            "filter_by_type",
            description="Used to only present items that have a specific type",
            options=["book", "electronics"]
        )
    )
)

In [12]:
filter_query = (
    base_query.similar(
        description_space,
        description_similar_param,
        sl.Param("description_similar_clause_weight"),
    )
    .filter(
        product.category
        == sl.Param(
            "filter_by_category",
            description="Used to only present items that have a specific category",
            options=["psychology","children","headphones","electronics","book"],
        )
    )
    .filter(
        product.review_rating
        >= sl.Param(
            "review_rating_bigger_than",
            description="Used to find items with a review rating bigger than the provided number.",
        )
    )
    .filter(
        product.price
        <= sl.Param(
            "price_smaller_than",
            description="Used to find items with a price smaller than the provided number.",
        )
    )
)

In [13]:
results = app.query(
    filter_query,
    filter_by_type="book",
    filter_by_category="psychology",
    review_rating_bigger_than=4.0,
    price_smaller_than=60.0,
    limit=3,
    query_description= "harry potter"
)

In [14]:
results.to_pandas()

,type,category,title,description,price,review_rating,id,similarity_score,rank
0,book,psychology,Psychology Basics,introductory psychology book for beginners and...,30.0,4.1,1,0.0,0
1,book,psychology,Mindfulness for Everyday Life,"mindfulness, stress reduction, mental health a...",55.0,4.7,4,0.0,1


## Semantic query

In [15]:

semantic_query = (
    base_query.similar(
        description_space,
        description_similar_param,
        sl.Param("description_similar_clause_weight"),
    )
    .similar(
        title_space,
        title_similar_param,
        sl.Param("title_similar_clause_weight"),
    )
    .filter(
        product.category
        == sl.Param(
            "filter_by_category",
            description="Used to only present items that have a specific cateogry",
            options=["psychology","children","headphones","electronics","book"],
        )
    )
)

similar_items_query = semantic_query.with_vector(product, sl.Param("product_id"))

In [24]:

results = app.query(
    semantic_query,
    filter_by_type="book",
    query_description="harry potter",
    query_title = "book",
    description_weight = 1,
    title_weight = 1,
    limit=3,
    review_rating_maximizer_weight = 0,
    price_minimizer_weights = 1
)

In [25]:
results.knn_params

{'title_weight': 1.0,
 'description_weight': 1.0,
 'review_rating_maximizer_weight': 0.0,
 'price_minimizer_weights': 1.0,
 'limit': 3,
 'natural_query': None,
 'filter_by_type': 'book',
 'query_description': 'harry potter',
 'query_title': 'book',
 'filter_by_category': None,
 'radius_param': None,
 'description_similar_clause_weight': 1.0,
 'title_similar_clause_weight': 1.0}

In [26]:
results.to_pandas()

,type,category,title,description,price,review_rating,id,similarity_score,rank
0,book,children,Cheap Kids Learning Book,children learning book with basic emotional sk...,12.0,3.2,3,0.557365,0
1,book,psychology,Psychology Basics,introductory psychology book for beginners and...,30.0,4.1,1,0.507395,1
2,book,psychology,Mindfulness for Everyday Life,"mindfulness, stress reduction, mental health a...",55.0,4.7,4,0.384498,2
